In [ ]:
# ⚠️ RUN THIS ONCE to force-kill the old runtime.
# After this, the kernel will die. Then:
#   1. Click kernel selector (top-right) → reconnect to "Google Colab"
#   2. A NEW runtime with H100 GPU will spin up
#   3. DELETE this cell, then run from Cell 1 onwards
import os
os.kill(os.getpid(), 9)

# KaleidoNet: Full Training on Colab Pro (TPU v6e-1)

This notebook runs the full KaleidoNet experiment suite on TPU v6e-1.

**Experiments:**
1. KaleidoNet (10K steps, 3 seeds) on CIFAR-100
2. Dense ViT baseline (10K steps, 3 seeds) on CIFAR-100
3. Pruning baselines
4. Pillar ablation study
5. Sparsity sweep

**Estimated time:** ~2-3 hours total on TPU

In [3]:
# Check hardware and ensure correct PyTorch
import subprocess, sys, os, shutil

# Check GPU via nvidia-smi
nvidia_smi = shutil.which('nvidia-smi')
if nvidia_smi:
    os.system('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader')
    HW = 'gpu'
else:
    print("nvidia-smi not found")
    HW = None

# Check PyTorch CUDA support
import torch
if torch.cuda.is_available():
    device = torch.device('cuda')
    HW = 'gpu'
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
elif HW == 'gpu':
    # GPU exists but PyTorch is CPU-only — install CUDA build
    print(f"PyTorch {torch.__version__} is CPU-only, installing CUDA build...")
    os.system(f'{sys.executable} -m pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu124')
    print("CUDA PyTorch installed. Please RESTART THE KERNEL and re-run this cell.")
    device = None
else:
    # Try detecting TPU
    try:
        import torch_xla.core.xla_model as xm
        device = xm.xla_device()
        HW = 'tpu'
        print(f"TPU device: {device}")
    except Exception:
        device = torch.device('cpu')
        HW = 'cpu'
        print("No GPU or TPU. Using CPU.")
        print("If you expected H100, go to Runtime > Change runtime type > GPU A100/H100, then restart.")

print(f"\nPyTorch: {torch.__version__}")
print(f"Hardware: {HW}")

nvidia-smi not found
No GPU or TPU. Using CPU.
If you expected H100, go to Runtime > Change runtime type > GPU A100/H100, then restart.

PyTorch: 2.9.0+cpu
Hardware: cpu


/tmp/ipykernel_430301/1273964957.py:30: DeprecationWarning: Use torch_xla.device instead
  device = xm.xla_device()


In [2]:
# Setup: Clone KaleidoNet from GitHub
import os, sys

ON_COLAB = os.path.exists('/content')

if ON_COLAB:
    os.chdir('/content')
    
    if os.path.exists('/content/KaleidoNet/kaleidonet/__init__.py'):
        print('KaleidoNet already present, skipping clone')
    else:
        print('Cloning KaleidoNet from GitHub...')
        os.system('git clone https://github.com/karimmagdy/kaleidonet-colab-src.git KaleidoNet')
        assert os.path.exists('/content/KaleidoNet/kaleidonet/__init__.py'), 'Clone failed!'
        print('Clone successful!')
    
    os.chdir('/content/KaleidoNet')
    sys.path.insert(0, '/content/KaleidoNet')
else:
    if not os.path.exists('kaleidonet'):
        print('Not in KaleidoNet directory!')

print(f'CWD: {os.getcwd()}')
print(f'Key files: {[f for f in os.listdir(".") if f.endswith(".py") or f == "kaleidonet" or f == "configs"]}')


Cloning KaleidoNet from GitHub...
Clone successful!
CWD: /content/KaleidoNet
Key files: ['configs', 'run.py', 'kaleidonet']


In [3]:
# Install dependencies
!pip install -e '.[full]' -q
!pip install pyyaml -q

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
ERROR: Exception:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 377, in run
    requirement_set = resolver.resolve(
                      ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/resolution/resolvelib/resolver.py", line 76, in resolve
    collected = self.factory.collect_root_requirements(root_reqs)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/

In [4]:
# Verify installation
import sys
sys.path.insert(0, '.')
from kaleidonet.model import KaleidoNet
model = KaleidoNet()
n_params = sum(p.numel() for p in model.parameters())
print(f'KaleidoNet params: {n_params:,}')
print('Import OK')

KaleidoNet params: 43,492,506
Import OK


In [11]:
# Pull latest code with TPU/XLA support (force-reset to handle any local conflicts)
import os, sys
os.chdir('/content/KaleidoNet')
os.system('git fetch origin && git reset --hard origin/main')

# Set XLA env vars for BF16 performance on TPU
os.environ["XLA_USE_BF16"] = "1"

# Clear cached modules so Python picks up the updated code
for mod_name in list(sys.modules.keys()):
    if 'kaleidonet' in mod_name:
        del sys.modules[mod_name]

# Verify: check device detection
from kaleidonet.training.trainer import TrainerConfig
cfg = TrainerConfig()
print(f"Device: {cfg.device}")
if "xla" in cfg.device:
    print("TPU will be used for all training!")
else:
    print(f"WARNING: Device is {cfg.device}, not TPU")

Device: xla:0
TPU will be used for all training!


## 1. Main Experiments: KaleidoNet + Dense ViT (3 seeds each)

In [12]:
# KaleidoNet on CIFAR-100 — 3 seeds × 10K steps
# This is the PRIMARY experiment for the paper
!python experiments/multi_seed_run.py \
    --model kaleidonet \
    --dataset cifar100 \
    --seeds 1 2 3 \
    --steps 10000


  KaleidoNet cifar100 seed=1 | seed=1 | steps=10000
  cmd: /usr/bin/python3 -u run.py experiments/baselines/train_cifar100.py --seed 1 --steps 10000

KaleidoNet CIFAR-100 Training
Seed: 1
Total params:  5,490,752
Active params: 5,490,752
Active fraction: 100.0%

Dense FLOPs (per sample):   682,444,800
Active FLOPs at init:       239,516,160
FLOPs budget (50% active):  119,758,080
Target: 50% of init active = 18% of dense

Seed set to 1
Starting KaleidoNet training on cpu
  Model params: 5,490,752
  FLOPs budget: 119,758,080
  Max steps: 10000

Step      0 | loss=4.8002 | task=4.7367 | soft=62% hard=100% | logits=[0.50,0.50,0.50] | λ=0.0100 | τ=1.000 | lr=1.20e-05 | 4.2 steps/s
Step     50 | loss=4.7794 | task=4.7252 | soft=62% hard=100% | logits=[0.46,0.50,0.53] | λ=0.0100 | τ=0.982 | lr=4.08e-05 | 5.1 steps/s
Step    100 | loss=4.5753 | task=4.5252 | soft=62% hard=100% | logits=[0.44,0.50,0.58] | λ=0.0100 | τ=0.964 | lr=1.14e-04 | 5.0 steps/s
Step    150 | loss=4.5152 | task=4.4638 |

In [13]:
# Dense ViT baseline on CIFAR-100 — 3 seeds × 10K steps
!python experiments/multi_seed_run.py \
    --model dense \
    --dataset cifar100 \
    --seeds 1 2 3 \
    --steps 10000


  Dense ViT cifar100 seed=1 | seed=1 | steps=10000
  cmd: /usr/bin/python3 -u experiments/baselines/dense_vit_baseline.py --seed 1 --steps 10000

Dense ViT Baseline — CIFAR-100
Seed: 1
/content/KaleidoNet/experiments/baselines/dense_vit_baseline.py:110: DeprecationWarning: Use torch_xla.device instead
  device = str(xm.xla_device())
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(
Total params: 1,821,028
Device: cpu
Step      0 | loss=4.7973 | lr=3.00e-04 | 8.1 steps/s
Step     50 | loss=4.2872 | lr=3.00e-04 | 10.0 steps/s
Step    100 | loss=4.0293 | lr=3.00e-04 | 10.3 steps/s
Step    150 | loss=4.2374 | lr=3.00e-04 | 10.4 steps/s
Step    200 | loss=4.1086 | lr=3.00e-04 | 10.5 steps/s
Step    250 | loss=3.9341 | lr=3.00e-04 | 10.5 steps/s
Step    300 | loss=4.0867 | lr=2.99e-04 | 10.4 steps/s
Step    350 | loss=4.2699 | l

## 2. Additional Datasets

In [14]:
# CIFAR-10: KaleidoNet (3 seeds)
!python experiments/multi_seed_run.py \
    --model kaleidonet \
    --dataset cifar10 \
    --seeds 1 2 3 \
    --steps 10000


  KaleidoNet cifar10 seed=1 | seed=1 | steps=10000
  cmd: /usr/bin/python3 -u run.py experiments/baselines/train_cifar10.py --seed 1 --steps 10000

KaleidoNet CIFAR-10 Training
Seed: 1
Total params:  5,473,382
Active params: 5,473,382
Active fraction: 100.0%

Dense FLOPs (per sample):   680,232,960
Active FLOPs at init:       237,269,760
FLOPs budget (50% active):  118,634,880

Seed set to 1
Starting KaleidoNet training on cpu
  Model params: 5,473,382
  FLOPs budget: 118,634,880
  Max steps: 10000

Step      0 | loss=2.4605 | task=2.3926 | soft=62% hard=100% | logits=[0.50,0.50,0.50] | λ=0.0100 | τ=1.000 | lr=1.20e-05 | 5.5 steps/s
Step     50 | loss=2.3607 | task=2.3018 | soft=62% hard=100% | logits=[0.46,0.50,0.54] | λ=0.0100 | τ=0.982 | lr=4.08e-05 | 6.3 steps/s
Step    100 | loss=2.0562 | task=2.0051 | soft=62% hard=100% | logits=[0.43,0.51,0.58] | λ=0.0100 | τ=0.964 | lr=1.14e-04 | 6.1 steps/s
Step    150 | loss=2.1335 | task=2.0833 | soft=62% hard=100% | logits=[0.43,0.51,0.61]

: 

In [ ]:
# CIFAR-10: Dense ViT baseline (3 seeds)
!python experiments/multi_seed_run.py \
    --model dense \
    --dataset cifar10 \
    --seeds 1 2 3 \
    --steps 10000

## 3. Pruning Baselines

In [ ]:
# Run pruning baselines (magnitude, random, lottery ticket)
!python run.py experiments/baselines/pruning_baselines.py

## 4. Ablation Studies

In [ ]:
# Pillar ablation (toggle each mechanism on/off)
!python run.py experiments/ablations/pillar_ablation.py

In [ ]:
# Sparsity sweep (vary target sparsity)
!python run.py experiments/ablations/sparsity_sweep.py

## 5. Collect and Analyze Results

In [ ]:
# Analyze multi-seed results
!python experiments/analyze_multi_seed.py

In [ ]:
# List all result files
import glob
results = sorted(glob.glob('results/**/*.json', recursive=True))
print(f'Found {len(results)} result files:')
for r in results:
    print(f'  {r}')

In [ ]:
# Download results to local machine
import shutil
shutil.make_archive('kaleidonet_results', 'zip', '.', 'results')
from google.colab import files
files.download('kaleidonet_results.zip')
print('Results downloaded!')

## 6. Inference Benchmarks

In [ ]:
# Benchmark inference speed and FLOPs
!python run.py experiments/benchmarks/benchmark_inference.py